In [1]:
import torch
from diffusers import StableDiffusionPipeline
import warnings
warnings.filterwarnings("ignore")

print("⚙️ Работаем на CPU")

model_id = "segmind/tiny-sd"

print("⏳ Загрузка модели (первый раз скачается ~1 ГБ)...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    safety_checker=None,
)
pipe = pipe.to("cpu")

prompt = "Young anthropomorphic fluffy brown-eyed grey raccoon " \
"in yellow t-shirt in light blue spacious working jeans with suspenders and white straw " \
"bowler hat is walking through the yellow wheat field, colored pencil cartoon style"
print("🎨 Генерация на CPU (это займет 1-3 минуты)...")

image = pipe(
    prompt,
    num_inference_steps=15,
    guidance_scale=7.5,
    width=384,
    height=384,
).images[0]

image.save("cabin_cpu.png")
print("✅ Готово! Файл cabin_cpu.png создан.")

[transformers] `CLIPImageProcessor` requires torchvision (not installed); falling back to `CLIPImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `CLIPImageProcessorPil` directly to silence this warning.
[transformers] `SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.


⚙️ Работаем на CPU
⏳ Загрузка модели (первый раз скачается ~1 ГБ)...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

An error occurred while trying to fetch C:\Users\ASUS\.cache\huggingface\hub\models--segmind--tiny-sd\snapshots\cad0bd7495fa6c4bcca01b19a723dc91627fe84f\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\ASUS\.cache\huggingface\hub\models--segmind--tiny-sd\snapshots\cad0bd7495fa6c4bcca01b19a723dc91627fe84f\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch C:\Users\ASUS\.cache\huggingface\hub\models--segmind--tiny-sd\snapshots\cad0bd7495fa6c4bcca01b19a723dc91627fe84f\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\ASUS\.cache\huggingface\hub\models--segmind--tiny-sd\snapshots\cad0bd7495fa6c4bcca01b19a723dc91627fe84f\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

🎨 Генерация на CPU (это займет 1-3 минуты)...


  0%|          | 0/15 [00:00<?, ?it/s]

✅ Готово! Файл cabin_cpu.png создан.


In [2]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import warnings

warnings.filterwarnings("ignore")

print("⏳ Загрузка модели BLIP...")
model_id = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float32
).to("cpu")

print("🖼️ Чтение изображения...")
try:
    image = Image.open("cabin_cpu.png").convert("RGB")
    # Быстрая проверка: если файл пустой или битый, модель сойдет с ума
    print(f"Размер изображения: {image.size}") 
except FileNotFoundError:
    print("❌ Ошибка: файл cabin_cpu.png не найден.")
    image = None

if image:
    print("📝 Генерация описания (с защитой от повторов)...")
    
    inputs = processor(image, return_tensors="pt").to("cpu")
    
    # === ГЛАВНЫЕ ИСПРАВЛЕНИЯ ЗДЕСЬ ===
    out = model.generate(
        **inputs, 
        max_new_tokens=75,
        repetition_penalty=1.5,       # Штрафует модель за использование тех же слов
        no_repeat_ngram_size=3        # Запрещает повторять 3 слова подряд
    )
    
    caption = processor.decode(out[0], skip_special_tokens=True)

    print("\n" + "="*50)
    print(f"✅ ОПИСАНИЕ ИЗОБРАЖЕНИЯ:\n{caption}")
    print("="*50)

⏳ Загрузка модели BLIP...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

🖼️ Чтение изображения...
Размер изображения: (384, 384)
📝 Генерация описания (с защитой от повторов)...

✅ ОПИСАНИЕ ИЗОБРАЖЕНИЯ:
a painting of a rad in overalls and hat


In [3]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
import torch
import soundfile as sf
import warnings

warnings.filterwarnings("ignore")

print("⏳ Загрузка модели озвучки (SpeechT5)...")
tts_model_id = "microsoft/speecht5_tts"
vocoder_id = "microsoft/speecht5_hifigan"

processor = SpeechT5Processor.from_pretrained(tts_model_id)
tts_model = SpeechT5ForTextToSpeech.from_pretrained(tts_model_id, torch_dtype=torch.float32).to("cpu")
vocoder = SpeechT5HifiGan.from_pretrained(vocoder_id, torch_dtype=torch.float32).to("cpu")

# Берем описание, которое мы получили, или напишем более красивое вручную для теста
text_to_speak = "A cozy wooden cabin in a snowy forest." 
print(f"📝 Озвучиваем текст: '{text_to_speak}'")

# Подготовка входов
inputs = processor(text=text_to_speak, return_tensors="pt").to("cpu")

# SpeechT5 требует вектор голоса (speaker embeddings). 
# Для простоты используем нулевой тензор (стандартный нейтральный голос)
speaker_embeddings = torch.zeros((1, 512), dtype=torch.float32).to("cpu")

print("🔊 Генерация аудио на CPU (подождите 10-20 секунд)...")
with torch.no_grad():
    speech = tts_model.generate_speech(
        inputs["input_ids"], 
        speaker_embeddings, 
        vocoder=vocoder
    )

# Сохраняем в WAV файл (16000 Гц - стандарт для речи)
sf.write("image_description.wav", speech.cpu().numpy(), samplerate=16000)
print("✅ УСПЕХ! Аудио сохранено как 'image_description.wav'. Можешь его прослушать!")

⏳ Загрузка модели озвучки (SpeechT5)...


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

📝 Озвучиваем текст: 'A cozy wooden cabin in a snowy forest.'
🔊 Генерация аудио на CPU (подождите 10-20 секунд)...
✅ УСПЕХ! Аудио сохранено как 'image_description.wav'. Можешь его прослушать!


In [4]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa
import warnings
import os

warnings.filterwarnings("ignore")

# 1. Скачиваем тестовый аудиофайл, если его нет
audio_path = "test_audio.wav"
if not os.path.exists(audio_path):
    print("⏳ Скачиваю тестовый аудиофайл с речью...")
    import urllib.request
    urllib.request.urlretrieve("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac", audio_path)

print("⏳ Загрузка модели распознавания речи (Whisper-tiny)...")
asr_model_id = "openai/whisper-tiny"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id, torch_dtype=torch.float32).to("cpu")

# 2. Читаем аудио
print(f"🎤 Читаем файл: {audio_path}")
speech_array, sampling_rate = librosa.load(audio_path, sr=16000)

# 3. Распознаем речь
print("📝 Преобразую речь в текст...")
input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to("cpu")

with torch.no_grad():
    predicted_ids = asr_model.generate(input_features)

transcription = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(f"✅ Распознанный текст: '{transcription}'")

⏳ Загрузка модели распознавания речи (Whisper-tiny)...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

🎤 Читаем файл: test_audio.wav


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


📝 Преобразую речь в текст...


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


✅ Распознанный текст: ' I have a dream. Good one day. This nation will rise up. Live out the true meaning of its dream.'


In [5]:
import warnings
warnings.filterwarnings("ignore")

print("🎨 Генерирую изображение по распознанной речи...")

# Берем распознанный текст и добавляем немного "магии" для красоты картинки
prompt_from_audio = transcription + ", highly detailed, digital art, masterpiece"
print(f"📝 Промпт для генерации: '{prompt_from_audio}'")

# Если переменная `pipe` всё ещё в памяти (из Шага 1), используем её.
# Если ты перезапускал ядро, раскомментируй 3 строки ниже:
# from diffusers import StableDiffusionPipeline
# import torch
# pipe = StableDiffusionPipeline.from_pretrained("segmind/tiny-sd", torch_dtype=torch.float32, safety_checker=None).to("cpu")

image_from_audio = pipe(
    prompt_from_audio,
    num_inference_steps=15,      # Быстрая генерация для CPU
    guidance_scale=7.5,
    width=384,
    height=384,
).images[0]

image_from_audio.save("from_audio.png")
print("✅ УСПЕХ! Изображение по аудио сохранено как 'from_audio.png'")

# Если ты в Jupyter Notebook, раскомментируй строку ниже, чтобы увидеть картинку сразу:
# display(image_from_audio)

🎨 Генерирую изображение по распознанной речи...
📝 Промпт для генерации: ' I have a dream. Good one day. This nation will rise up. Live out the true meaning of its dream., highly detailed, digital art, masterpiece'


  0%|          | 0/15 [00:00<?, ?it/s]

✅ УСПЕХ! Изображение по аудио сохранено как 'from_audio.png'


In [6]:
import torch
import soundfile as sf
import warnings
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
)
import librosa

warnings.filterwarnings("ignore")

print("🚀 Загрузка легковесного пайплайна перевода (STT -> MT -> TTS)...")

# 1. STT: Whisper-tiny (Распознавание речи)
print("  ⏳ Загрузка Whisper-tiny...")
asr_model_id = "openai/whisper-tiny"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id, torch_dtype=torch.float32).to("cpu")

# 2. MT: OPUS-MT (Перевод текста Русский -> Английский)
print("  ⏳ Загрузка модели перевода (OPUS-MT ru-en)...")
mt_model_id = "Helsinki-NLP/opus-mt-ru-en"
mt_tokenizer = AutoTokenizer.from_pretrained(mt_model_id)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_model_id, torch_dtype=torch.float32).to("cpu")

# 3. TTS: SpeechT5 (Озвучка текста)
print("  ⏳ Загрузка SpeechT5 для озвучки...")
tts_model_id = "microsoft/speecht5_tts"
vocoder_id = "microsoft/speecht5_hifigan"
tts_processor = SpeechT5Processor.from_pretrained(tts_model_id)
tts_model = SpeechT5ForTextToSpeech.from_pretrained(tts_model_id, torch_dtype=torch.float32).to("cpu")
vocoder = SpeechT5HifiGan.from_pretrained(vocoder_id, torch_dtype=torch.float32).to("cpu")

print("✅ Все модели загружены!\n")

# --- ДЕМО-РЕЖИМ: Перевод и озвучка заданного текста ---
# (Если у тебя есть свой .wav файл с русской речью, замени russian_text на None 
#  и раскомментируй блок загрузки аудио ниже)

russian_text = "Привет! Это тест системы перевода речи в реальном времени. Нейросети работают отлично."

print(f"📝 Исходный текст (RU): '{russian_text}'")

# Шаг А: Перевод текста (Text-to-Text)
print("🔄 Перевожу текст на английский...")
inputs_mt = mt_tokenizer(russian_text, return_tensors="pt").to("cpu")
translation_ids = mt_model.generate(**inputs_mt, max_length=50)
english_text = mt_tokenizer.decode(translation_ids[0], skip_special_tokens=True)
print(f"✅ Переведенный текст (EN): '{english_text}'")

# Шаг Б: Озвучка переведенного текста (Text-to-Speech)
print("🔊 Озвучиваю переведенный текст...")
tts_inputs = tts_processor(text=english_text, return_tensors="pt").to("cpu")
speaker_embeddings = torch.zeros((1, 512), dtype=torch.float32).to("cpu") # Нейтральный голос

with torch.no_grad():
    speech = tts_model.generate_speech(
        tts_inputs["input_ids"], 
        speaker_embeddings, 
        vocoder=vocoder
    )

# Сохраняем результат
output_audio_path = "translated_speech.wav"
sf.write(output_audio_path, speech.cpu().numpy(), samplerate=16000)
print(f"🎉 УСПЕХ! Переведенная речь сохранена как '{output_audio_path}'. Можешь её прослушать!")

"""
# --- АЛЬТЕРНАТИВА: Если хочешь загрузить реальное русское аудио ---
# audio_path = "my_russian_voice.wav" # Укажи путь к своему файлу
# speech_array, sr = librosa.load(audio_path, sr=16000)
# input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to("cpu")
# predicted_ids = asr_model.generate(input_features, language="ru", task="transcribe")
# russian_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
# print(f"Распознано: {russian_text}")
# ...далее код идет так же, от Шага А (Перевод текста)
"""

🚀 Загрузка легковесного пайплайна перевода (STT -> MT -> TTS)...
  ⏳ Загрузка Whisper-tiny...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  ⏳ Загрузка модели перевода (OPUS-MT ru-en)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  ⏳ Загрузка SpeechT5 для озвучки...


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

✅ Все модели загружены!

📝 Исходный текст (RU): 'Привет! Это тест системы перевода речи в реальном времени. Нейросети работают отлично.'
🔄 Перевожу текст на английский...
✅ Переведенный текст (EN): 'Hey, it's a real-time speech system test.'
🔊 Озвучиваю переведенный текст...
🎉 УСПЕХ! Переведенная речь сохранена как 'translated_speech.wav'. Можешь её прослушать!


'\n# --- АЛЬТЕРНАТИВА: Если хочешь загрузить реальное русское аудио ---\n# audio_path = "my_russian_voice.wav" # Укажи путь к своему файлу\n# speech_array, sr = librosa.load(audio_path, sr=16000)\n# input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to("cpu")\n# predicted_ids = asr_model.generate(input_features, language="ru", task="transcribe")\n# russian_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]\n# print(f"Распознано: {russian_text}")\n# ...далее код идет так же, от Шага А (Перевод текста)\n'

In [15]:
import torch
import sounddevice as sd
import numpy as np
import soundfile as sf
import warnings
from datasets import load_dataset
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
)

warnings.filterwarnings("ignore")

# ==========================================
# НОВАЯ НАДЕЖНАЯ ФУНКЦИЯ: Загрузка голоса через datasets
# ==========================================
def get_speaker_embedding(speaker_index=100):
    """
    Загружает x-vector из официального датасета cmu-arctic-xvectors.
    speaker_index: 0-50 (преимущественно женские), 100-150 (преимущественно мужские).
    По умолчанию берем 100 (четкий мужской голос).
    """
    try:
        print("⏳ Загрузка датасета голосов (может занять ~30 сек при первом запуске)...")
        # Загружаем validation split, где лежат готовые 512-мерные векторы
        embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
        
        # Извлекаем вектор по индексу
        xvector_list = embeddings_dataset["xvector"]
        embedding = torch.tensor(xvector_list[speaker_index], dtype=torch.float32).unsqueeze(0)
        
        print(f"✅ Голос (индекс {speaker_index}) успешно загружен!")
        return embedding
    except Exception as e:
        print(f"⚠️ Не удалось загрузить датасет голосов ({e}). Используем нейтральный голос.")
        return torch.zeros((1, 512), dtype=torch.float32)

# ==========================================
# ЧАСТЬ 1: Функция записи с микрофона (VAD)
# ==========================================
def record_until_silence(filename="temp_input.wav", samplerate=16000, silence_threshold=0.015, silence_duration=1.5):
    print("\n🎤 СЛУШАЮ... Говорите в микрофон.")
    print("   (Программа автоматически остановит запись через 1.5 сек после окончания речи)")
    
    frames = []
    silence_counter = 0.0
    chunk_size = int(samplerate * 0.1)
    
    try:
        with sd.InputStream(samplerate=samplerate, channels=1, dtype='float32') as stream:
            while True:
                chunk, overflowed = stream.read(chunk_size)
                chunk = chunk.flatten()
                frames.append(chunk)
                
                energy = np.sqrt(np.mean(chunk**2))
                
                if energy < silence_threshold:
                    silence_counter += 0.1
                    if silence_counter >= silence_duration:
                        print("🛑 Тишина обнаружена. Обрабатываю...\n")
                        break
                else:
                    silence_counter = 0.0
                    
    except Exception as e:
        print(f"❌ Ошибка записи: {e}")
        return None
        
    audio_data = np.concatenate(frames)
    sf.write(filename, audio_data, samplerate)
    return filename

# ==========================================
# ЧАСТЬ 2: Загрузка моделей
# ==========================================
print("🚀 Загрузка моделей (подождите)...")
device = "cpu"

asr_model_id = "openai/whisper-tiny"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id, torch_dtype=torch.float32).to(device)

mt_model_id = "Helsinki-NLP/opus-mt-ru-en"
mt_tokenizer = AutoTokenizer.from_pretrained(mt_model_id)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_model_id, torch_dtype=torch.float32).to(device)

tts_model_id = "microsoft/speecht5_tts"
vocoder_id = "microsoft/speecht5_hifigan"
tts_processor = SpeechT5Processor.from_pretrained(tts_model_id)
tts_model = SpeechT5ForTextToSpeech.from_pretrained(tts_model_id, torch_dtype=torch.float32).to(device)
vocoder = SpeechT5HifiGan.from_pretrained(vocoder_id, torch_dtype=torch.float32).to(device)

# === ЗАГРУЖАЕМ КОНКРЕТНЫЙ ГОЛОС ===
# Попробуй index=100 (мужской) или index=0 (женский)
speaker_embeddings = get_speaker_embedding(speaker_index=100)
speaker_embeddings = speaker_embeddings.to(device)

print("✅ Модели готовы! Можете начинать говорить.\n(Для выхода из программы нажмите Ctrl+C)")

# ==========================================
# ЧАСТЬ 3: Главный цикл Real-time обработки
# ==========================================
try:
    while True:
        audio_path = record_until_silence()
        if not audio_path:
            continue
            
        # 2. Распознавание (STT)
        speech_array, sr = sf.read(audio_path)
        input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        
        with torch.no_grad():
            predicted_ids = asr_model.generate(input_features, language="ru", task="transcribe")
        russian_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        print(f"📝 Распознано (RU): '{russian_text}'")
        
        # 3. Перевод (MT)
        inputs_mt = mt_tokenizer(russian_text, return_tensors="pt").to(device)
        with torch.no_grad():
            translation_ids = mt_model.generate(
                **inputs_mt, 
                max_length=100, 
                num_beams=4,          
                length_penalty=1.0    
            )
        english_text = mt_tokenizer.decode(translation_ids[0], skip_special_tokens=True)
        print(f"🔄 Перевод (EN): '{english_text}'")
        
        # 4. Озвучка (TTS) с выбранным голосом
        tts_inputs = tts_processor(text=english_text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            speech = tts_model.generate_speech(
                tts_inputs["input_ids"], 
                speaker_embeddings, # <-- Здесь теперь реальный вектор голоса
                vocoder=vocoder
            )
            
        output_path = "realtime_translation.wav"
        sf.write(output_path, speech.cpu().numpy(), samplerate=16000)
        print(f"🔊 Озвучка сохранена в: {output_path}\n" + "-"*50)

except KeyboardInterrupt:
    print("\n👋 Программа остановлена пользователем. До встречи!")

🚀 Загрузка моделей (подождите)...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

⏳ Загрузка датасета голосов (может занять ~30 сек при первом запуске)...
⚠️ Не удалось загрузить датасет голосов (Dataset scripts are no longer supported, but found cmu-arctic-xvectors.py). Используем нейтральный голос.
✅ Модели готовы! Можете начинать говорить.
(Для выхода из программы нажмите Ctrl+C)

🎤 СЛУШАЮ... Говорите в микрофон.
   (Программа автоматически остановит запись через 1.5 сек после окончания речи)
🛑 Тишина обнаружена. Обрабатываю...

📝 Распознано (RU): ' Собой, ну-ка, ну-ка.'
🔄 Перевод (EN): 'Sorry, come on, come on.'
🔊 Озвучка сохранена в: realtime_translation.wav
--------------------------------------------------

🎤 СЛУШАЮ... Говорите в микрофон.
   (Программа автоматически остановит запись через 1.5 сек после окончания речи)
🛑 Тишина обнаружена. Обрабатываю...

📝 Распознано (RU): ' СМЕХ'
🔄 Перевод (EN): 'LAUGHTER'
🔊 Озвучка сохранена в: realtime_translation.wav
--------------------------------------------------

🎤 СЛУШАЮ... Говорите в микрофон.
   (Программа автома